# Using the RooJSONFactoryWSTool to generate HS3 JSON with RooFit

This notebook shows a complete round-trip workflow: build a RooFit model, export it to an HS3 (Harmonized Statistics Serialization Standard) JSON file, and import it back into a fresh workspace.

The standard is still evolving. If you have questions or feedback about how you would like to use these workspaces in your analysis, please contact simon.cello@cern.ch.


# 0. Software setup

[HS3](https://hep-statistics-serialization-standard.github.io/) is a standardized format for publishing and archiving statistical models. It uses `JSON` to describe the computational graph of a likelihood while remaining human-readable.

With the experimental `RooJSONFactoryWSTool` in `ROOT`, a workspace can be translated to this format with relatively little code. Because the tool is still under active development, it is a good idea to work with a recent `ROOT` build.

In Binder, the version check below is a quick sanity check before you start the tutorial.


In [ ]:
import ROOT

# Print the ROOT build information so Binder users can confirm the environment.
print("ROOT Version =", ROOT.gROOT.GetVersion(), ",ROOT Date =", ROOT.gROOT.GetGitDate())


ROOT Version = 6.39.01 ,ROOT Date = Mar 29 2026, 09:57:34


# 1. Create a simple RooFit workspace

RooFit provides a large toolbox for statistical model building. This tutorial keeps the physics model intentionally small so the focus stays on the export and import steps of `RooJSONFactoryWSTool`.

We will build a one-dimensional signal-plus-background model, generate toy data from it, and fit it once before exporting anything. That gives us a clear baseline for the round-trip test later on.


## 1.1 Define the observables and PDF components

The model below has a Gaussian signal component and an exponential background component. Both share the same observable `x`, and `sig_frac` controls their mixture in the final combined PDF.


In [ ]:
# Define the observable that will be sampled and fitted.
x = ROOT.RooRealVar("x", "x", 0, 5)

# Signal model: a Gaussian peak with floating mean and width.
mean = ROOT.RooRealVar("mean", "mean", 2, 0, 5)
sigma = ROOT.RooRealVar("sigma", "sigma", 1, 1e-5, 2)
gauss = ROOT.RooGaussian("gauss", "gauss", x, mean, sigma)

# Background model: an exponential that falls across the observable range.
delta = ROOT.RooRealVar("delta", "delta", -1, -5, 0)
exponential = ROOT.RooExponential("exponential", "exponential", x, delta)

# Fraction of the signal component in the combined model.
sig_frac = ROOT.RooRealVar("sig_frac", "sig_frac", 0.8, 0, 1)

# Build a signal-plus-background PDF from the two components.
model = ROOT.RooAddPdf(
    "model",
    "model",
    ROOT.RooArgList(gauss, exponential),
    ROOT.RooArgList(sig_frac),
)


## 1.2 Generate a toy dataset

Once the PDF is defined, we can sample events from it. This gives us a synthetic dataset that behaves like a small analysis input and can later be stored in the workspace as well.


In [ ]:
# Generate a toy dataset of 100000 events from the composite model.
data = model.generate(ROOT.RooArgSet(x), 100000)


## 1.3 Fit the model before exporting

Before writing anything to JSON, it is useful to fit the model once and inspect the result. This provides a reference point for checking that the imported workspace behaves the same way.


In [ ]:
# Create a RooPlot frame for the original model and generated dataset.
xframe1 = x.frame(Title="Model pdf with data")

# Overlay the toy data and the analytical PDF.
data.plotOn(xframe1)
model.plotOn(xframe1)

# Fit the model to the generated data and print the fitted parameters.
r = model.fitTo(data, ROOT.RooFit.Save(), PrintLevel=-1)
r.Print()


[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- using generic CPU library compiled with no vectorizations
[#1] INFO:Fitting -- Creation of NLL object took 1.2633 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_modelData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization

  RooFitResult: minimized FCN value: 110386, estimated distance to minimum: 1.82861e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MINIMIZE=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
          

# 2. Export the workspace to HS3 JSON

The `RooJSONFactoryWSTool` works on a `RooWorkspace`, so we first import the PDF and the dataset into a workspace container. The export step then writes a JSON representation to `tutorial.json`.


In [ ]:
# Collect the model and dataset in a RooWorkspace before exporting.
ws = ROOT.RooWorkspace()
ws.Import(model)
ws.Import(data)

# Export the workspace content to an HS3 JSON document on disk.
tool = ROOT.RooJSONFactoryWSTool(ws)
tool.exportJSON("tutorial.json")


True

[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooAddPdf::model
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooGaussian::gauss
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::x
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::mean
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sigma
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sig_frac
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooExponential::exponential
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::delta
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing dataset modelData


# 3. Import the JSON into a fresh workspace

To test the round-trip, create a brand-new `RooWorkspace` and populate it from the JSON file. This mimics how a downstream user would reconstruct the model in a separate session.


In [ ]:
# Start from a fresh workspace and rebuild it from the JSON file.
ws_new = ROOT.RooWorkspace()
tool_new = ROOT.RooJSONFactoryWSTool(ws_new)
tool_new.importJSON("tutorial.json")

# Retrieve the imported PDF and dataset by their workspace names.
model_new = ws_new.pdf("model")
data_new = ws_new.data("modelData")


[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::delta
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::mean
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sig_frac
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::sigma
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing RooRealVar::x
[#1] INFO:ObjectHandling -- RooWorkspace::import() importing dataset modelData


## 3.1 Validate the round-trip

Now retrieve the imported PDF and dataset from the new workspace, fit them again, and compare the parameter values with the original fit. Small numerical differences are normal, but the overall result should be consistent.


In [ ]:
# Plot and refit the imported objects to validate the round-trip conversion.
xframe2 = x.frame(Title="Reimported model pdf with data")
data_new.plotOn(xframe2)
model_new.plotOn(xframe2)
r_new = model_new.fitTo(data_new, ROOT.RooFit.Save(), PrintLevel=-1)
r_new.Print()


[#1] INFO:Fitting -- RooAbsPdf::fitTo(model) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 1.32375 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_model_modelData) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: deactivating const optimization

  RooFitResult: minimized FCN value: 110386, estimated distance to minimum: 1.09752e-05
                covariance matrix quality: Full, accurate covariance matrix
                Status : MINIMIZE=0 HESSE=0 

    Floating Parameter    FinalValue +/-  Error   
  --------------------  --------------------------
                 delta   -9.5627e-01 +/-  3.73e-02
                  mean    9.8668e-01 

# 4. Compare the original and imported objects visually

A side-by-side plot makes the round-trip easier to inspect in Binder. The left panel uses the original workspace content, while the right panel uses the objects reconstructed from the JSON file.


In [ ]:
# Draw the original and reimported results side by side for comparison.
ROOT.gROOT.SetBatch(True)
c = ROOT.TCanvas("tutorial", "tutorial", 800, 400)
c.Divide(2)

# Left panel: the original model and generated data.
c.cd(1)
ROOT.gPad.SetLeftMargin(0.15)
xframe1.GetYaxis().SetTitleOffset(1.6)
xframe1.Draw()

# Right panel: the objects reconstructed from tutorial.json.
c.cd(2)
ROOT.gPad.SetLeftMargin(0.15)
xframe2.GetYaxis().SetTitleOffset(1.6)
xframe2.Draw()

c.Update()
display(c)


## What to look at next

After running the notebook, open `tutorial.json` to inspect the serialized model. In a Binder setting this file becomes a concrete artifact that users can download, compare, or use as a starting point for their own HS3 studies.
